# 05 — Menjalankan Role 3 NER Web App di Google Colab

Notebook ini menjalankan paket aplikasi Role 3 yang mencakup backend FastAPI, halaman web, test API, dan model spaCy dari Role 2.

**Sebelum mulai:** unduh file `Role3_Article_NER_Service.zip` dari hasil yang dibagikan di chat. ZIP berisi model supaya demo bisa segera dijalankan. Jangan unggah dataset mentah ke repository publik.

Jalankan setiap sel secara berurutan. URL proxy Colab bersifat sementara selama runtime aktif.

## 1. Unggah paket aplikasi

In [ ]:
from google.colab import files
from IPython.display import display, HTML
from pathlib import Path
import io, os, sys, json, zipfile, shutil, subprocess, time

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if not zip_name:
    raise ValueError('Unggah Role3_Article_NER_Service.zip terlebih dahulu.')

PROJECT = Path('/content/ner_service_project')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
    archive.extractall(PROJECT)

required_files = [
    PROJECT/'app'/'main.py',
    PROJECT/'web'/'index.html',
    PROJECT/'requirements.txt',
    PROJECT/'models'/'ner_custom_v1'/'config.cfg'
]
missing = [str(p.relative_to(PROJECT)) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError(f'Paket ZIP tidak lengkap, file yang hilang: {missing}')
print('Paket berhasil dibuka:', PROJECT)
print('File aplikasi dan model ditemukan.')

## 2. Instal dependensi

In [ ]:
install = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT/'requirements.txt')],
    text=True, capture_output=True
)
if install.returncode != 0:
    print(install.stdout)
    print(install.stderr[-6000:])
    raise RuntimeError('Instal dependensi gagal.')
print('Dependensi berhasil dipasang.')

## 3. Jalankan test API otomatis

In [ ]:
test_run = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', str(PROJECT/'tests')],
    cwd=str(PROJECT), text=True, capture_output=True
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr[-3000:])
if test_run.returncode != 0:
    raise RuntimeError('Ada tes yang gagal. Perbaiki sebelum demo.')
print('Semua test API lulus.')

## 4. Jalankan API dan halaman web

In [ ]:
import requests

log_path = '/content/ner_service_uvicorn.log'
log_handle = open(log_path, 'w', encoding='utf-8')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=str(PROJECT), stdout=log_handle, stderr=subprocess.STDOUT
)

health = None
for _ in range(40):
    try:
        response = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if response.ok:
            health = response.json()
            break
    except Exception:
        time.sleep(1)

if health is None:
    log_handle.flush()
    print(Path(log_path).read_text(encoding='utf-8')[-6000:])
    raise RuntimeError('Server tidak berhasil aktif. Periksa log di atas.')
print('Server aktif:', health)

from google.colab.output import eval_js
proxy_url = eval_js('google.colab.kernel.proxyPort(8000)')
display(HTML(
    '<div style="border:1px solid #bfdbfe;border-radius:12px;padding:16px">'
    '<h3>Article NER Service aktif</h3>'
    f'<p><a href="{proxy_url}" target="_blank" rel="noopener">Buka halaman web aplikasi</a></p>'
    f'<p><a href="{proxy_url}/docs" target="_blank" rel="noopener">Buka dokumentasi API</a></p>'
    '</div>'
))

## 5. Kirim contoh request dan cek offset

In [ ]:
payload = {
    'title': 'Malaysia and Maybank Announce New Plans',
    'body': 'Maybank said the company would expand its services in Kuala Lumpur on October 22, 2026.'
}
response = requests.post('http://127.0.0.1:8000/extract-entities', json=payload, timeout=30)
print('HTTP status:', response.status_code)
data = response.json()
display(data)

full_text = payload['title'].strip() + '\n' + payload['body'].strip()
for entity in data.get('entities', []):
    assert full_text[entity['start']:entity['end']] == entity['text']
print('Pemeriksaan offset karakter: lulus.')

## 6. Checklist demo

- Isi artikel pada halaman web dan tekan **Analisis entitas**.
- Tunjukkan tabel `text`, `label`, `start`, dan `end`.
- Uji judul kosong, isi kosong, dan artikel yang terlalu panjang.
- Tunjukkan hasil `pytest` dan dokumentasi API di `/docs`.
- URL web dari Colab hanya aktif selama runtime berjalan; siapkan ulang bila runtime terputus.

Catatan: hasil prediksi bisa salah. Metrik dari paket Role 2 harus diverifikasi karena tabel keseluruhan dan tabel per-label yang diberikan tidak konsisten.

In [ ]:
server.terminate()
print('Server dihentikan. Untuk menjalankan lagi, ulangi sel server pada tahap 4.')